# Argument Structure Audit — Interactive Notebook

Interactive exploration layer on top of the audit toolchain.  
Complements `report.py` (automated pipeline) and `query.py` (CLI queries).

**Usage:** set `DOCUMENT` in the Setup cell, then run cells top to bottom or jump to any section.

In [ ]:
import sys
from pathlib import Path

# --- Configure this path before running ---
DOCUMENT = "path/to/document.md"  # set this to your document path

_here = Path(".").resolve()
_tools = _here / "argument_structure_audit" / "tools"
sys.path.insert(0, str(_tools if (_tools / "extract_graph.py").exists() else _here))
from extract_graph import parse_document, build_graph, scan_declared_types
from t1_check import collect_results
from query import _parse_refs, _split_citation
import networkx as nx

_doc_path = Path(DOCUMENT)
if not _doc_path.exists():
    raise FileNotFoundError(f"Document not found: {_doc_path.resolve()}")
nodes, edges = parse_document(DOCUMENT)
G = build_graph(nodes, edges)
text = Path(DOCUMENT).read_text(encoding="utf-8")
declared = scan_declared_types(text)

print(f"Loaded: {Path(DOCUMENT).name}")
print(f"Nodes: {len(nodes)}  |  Edges: {len(edges)}")
print(f"Declared types: {sorted(declared)}")

## T1 Checks

Which checks pass, fail, or escalate? Use this to decide where to focus the audit session.

In [ ]:
rel = Path(DOCUMENT).name
check_results = collect_results(text.splitlines(), rel)

print(f"{'Check':<16} {'Result':<12} Notes")
print("-" * 60)
for name, (findings, counts) in check_results.items():
    fails     = [f for f in findings if f.result == "Fail"]
    escalated = [f for f in findings if f.result == "Escalated"]
    if not fails and not escalated:
        result, notes = "Pass", ""
    elif escalated and not fails:
        result = "Escalated"
        notes = f"{len(escalated)} items"
    else:
        result = "Fail"
        notes = f"{counts.get('CRITICAL',0)} critical, {counts.get('SIGNIFICANT',0)} significant"
    print(f"{name:<16} {result:<12} {notes}")

In [ ]:
# Drill into one check — change CHECK_NAME to inspect a different check
CHECK_NAME = "TYPE-LABEL"

if CHECK_NAME not in check_results:
    print(f"Unknown check: {CHECK_NAME!r}. Available: {list(check_results)}")
else:
    findings, counts = check_results[CHECK_NAME]
    print(f"{CHECK_NAME}: {len(findings)} finding(s)")
    for f in findings:
        print(f"  [{f.severity}] line {f.line}: {f.note[:90]}")

## Section Density

Which sections carry the most argument load? High Argument counts mean heavy justification work; high Unknown counts flag encoding gaps.

In [ ]:
top_headings = [
    n for n in nx.topological_sort(G)
    if G.nodes[n].get("type") == "heading" and G.nodes[n].get("level") == 2
]

print(f"{'Section':<50} {'Arg':>5} {'Scope':>5} {'Claim':>5} {'Closure':>7} {'Unk':>5} {'Total':>5}")
print("-" * 90)
for h in top_headings:
    desc = nx.descendants(G, h)
    ct = {k: 0 for k in ("Argument", "Scope", "Claim", "Closure", "Unknown")}
    for d in desc:
        key = G.nodes[d].get("content_type", "Unknown")
        ct[key] = ct.get(key, 0) + 1
    label = G.nodes[h].get("lead_sentence", h)[:50]
    total = sum(ct.values())
    print(f"{label:<50} {ct['Argument']:>5} {ct['Scope']:>5} {ct['Claim']:>5} {ct['Closure']:>7} {ct['Unknown']:>5} {total:>5}")

## Structural Health

Orphans, cycles, and the longest reasoning chain. Any orphan or cycle is a structural violation.

In [ ]:
orphans = [
    n for n in G.nodes()
    if G.in_degree(n) == 0 and G.nodes[n].get("type") != "heading"
]
print(f"Orphans: {len(orphans) if orphans else 'none'}")
for n in orphans:
    attrs = G.nodes[n]
    print(f"  {n}  line {attrs.get('line',0)}")

print(f"DAG: {'yes' if nx.is_directed_acyclic_graph(G) else 'NO — cycle detected'}")

try:
    chain = nx.dag_longest_path(G)
    print(f"Longest reasoning chain: {len(chain)} nodes")
    for i, n in enumerate(chain, 1):
        attrs = G.nodes[n]
        label = (attrs.get("topic") or attrs.get("lead_sentence", ""))[:60]
        print(f"  {i:>2}. {n:<50}  {label}")
except nx.NetworkXUnfeasible:
    print("Longest chain: undefined — graph contains cycles")

## Section Subgraph

Inspect the full argument tree for one section. Change `SLUG` to target a different heading.

In [ ]:
SLUG = "h2:1----"  # substring of a heading node ID

candidates = [n for n in G.nodes() if G.nodes[n].get("type") == "heading" and SLUG in n]
if not candidates:
    h2_headings = [n for n in G.nodes() if G.nodes[n].get("type") == "heading" and G.nodes[n].get("level") == 2]
    print("No match. Available h2 headings:")
    for h in h2_headings:
        print(f"  {h}")
elif len(candidates) > 1:
    print(f"Ambiguous — {len(candidates)} matches. Refine SLUG and re-run:")
    for c in candidates:
        print(f"  {c}")
else:
    root = candidates[0]
    scope = {root} | nx.descendants(G, root)
    sub = G.subgraph(scope)
    print(f"Subgraph: {root}  ({len(scope)} nodes)")
    print(f"{'Node ID':<50} {'Type':<18} {'Content-type':<14} {'Line':>4}  Topic")
    print("-" * 110)
    for n in nx.topological_sort(sub):
        attrs = sub.nodes[n]
        label = (attrs.get("topic") or attrs.get("lead_sentence", ""))[:40]
        print(f"{n:<50} {attrs.get('type',''):<18} {attrs.get('content_type',''):<14} {attrs.get('line',0):>4}  {label}")

## Delta-Audit Scope

Given a changed node, what is the full audit scope?  
- **Ancestors** — upstream nodes whose conclusions may now be invalidated  
- **Descendants** — downstream nodes that depend on the changed node  
- **Prose citations** — sections that cite this item via `[→ ...]` (logical blast radius not captured by the graph)

Change `NODE_ID` to a full or partial node ID. Set `REF_SLUG` to the citation text to search (can differ from node ID — use the label as it appears in prose).

In [ ]:
NODE_ID  = ""   # full or partial node ID — run density or orphans first to find IDs
REF_SLUG = ""   # citation text to search in prose (e.g. "OQ-EC.4", "§5 Argument (source orthogonality)")

resolved_node = None

if not NODE_ID:
    print("Set NODE_ID to a node ID or partial match. Run 'density' to see heading IDs.")
elif NODE_ID in G:
    resolved_node = NODE_ID
else:
    matches = [n for n in G.nodes() if NODE_ID in n]
    if not matches:
        print(f"Node '{NODE_ID}' not found.")
    elif len(matches) > 1:
        print(f"Ambiguous — {len(matches)} matches. Refine NODE_ID and re-run:")
        for m in matches:
            print(f"  {m}")
    else:
        resolved_node = matches[0]

if resolved_node:
    ancs = nx.ancestors(G, resolved_node)
    desc = nx.descendants(G, resolved_node)
    print(f"Node: {resolved_node}")
    print(f"Ancestors (upstream invalidation): {len(ancs)}")
    print(f"Descendants (downstream propagation): {len(desc)}")
    print(f"Structural delta-audit scope: {len(ancs) + len(desc) + 1} nodes (including self)")
    print()
    print("--- Ancestors ---")
    for n in sorted(ancs, key=lambda x: G.nodes[x].get("line", 0)):
        attrs = G.nodes[n]
        label = (attrs.get("topic") or attrs.get("lead_sentence", ""))[:50]
        print(f"  {n:<50}  line {attrs.get('line',0):>4}  {label}")
    print()
    print("--- Descendants ---")
    for n in sorted(desc, key=lambda x: G.nodes[x].get("line", 0)):
        attrs = G.nodes[n]
        label = (attrs.get("topic") or attrs.get("lead_sentence", ""))[:50]
        print(f"  {n:<50}  line {attrs.get('line',0):>4}  {label}")

if REF_SLUG:
    inverted = _parse_refs(DOCUMENT)
    ref_matches = {t: locs for t, locs in inverted.items() if REF_SLUG.lower() in t.lower()}
    if not ref_matches:
        print(f"\nProse citations: no [→ ...] citations matching '{REF_SLUG}' found.")
    else:
        total = sum(len(locs) for locs in ref_matches.values())
        sections = {section for locs in ref_matches.values() for _, section in locs}
        print(f"\n--- Prose citations matching '{REF_SLUG}' ({total} occurrence(s) in {len(sections)} section(s)) ---")
        for tgt, locs in sorted(ref_matches.items()):
            for lineno, section in sorted(locs):
                print(f"  line {lineno:>4}  {section}")

## Prose Cross-Reference Index

Full matrix of `[→ §N]` and `[→ OQ-EC.N]` prose citations — sections as rows, citation targets as values. Use this to see all logical dependencies that the structural graph does not encode.

Set `REF_TARGET` to an empty string for the full matrix, or to a substring to get the inverted lookup for one item.

In [ ]:
REF_TARGET = ""  # leave empty for full matrix; set to e.g. "OQ-EC.4" for inverted lookup

inverted = _parse_refs(DOCUMENT)

if not inverted:
    print("No prose cross-references found.")
elif REF_TARGET:
    matches = {t: locs for t, locs in inverted.items() if REF_TARGET.lower() in t.lower()}
    if not matches:
        print(f"No citations found matching '{REF_TARGET}'.")
    else:
        total = sum(len(locs) for locs in matches.values())
        print(f"Citations matching '{REF_TARGET}': {total} occurrence(s) across {len(matches)} target(s)")
        print()
        for tgt, locs in sorted(matches.items()):
            print(f"  Target: [{tgt}]")
            for lineno, section in sorted(locs):
                print(f"    line {lineno:>4}  in: {section}")
else:
    by_section: dict = {}
    for tgt, locs in inverted.items():
        for lineno, section in locs:
            by_section.setdefault(section, []).append((lineno, tgt))
    for section in by_section:
        by_section[section].sort()
    section_order = sorted(by_section.keys(), key=lambda s: by_section[s][0][0])
    total_citations = sum(len(v) for v in by_section.values())
    print(f"Prose cross-references: {total_citations} citation(s) to {len(inverted)} distinct target(s)")
    print()
    for section in section_order:
        cites = by_section[section]
        print(f"  {section}  ({len(cites)} citation(s))")
        for lineno, tgt in cites:
            print(f"    line {lineno:>4}  → {tgt}")
        print()

## Shared Premises — Trunk Detection

Reports citation targets cited from N or more distinct sections. High-N entries are shared-premise candidates: falsifying one removes a premise from every citing section simultaneously.

`SHARED_MIN` sets the threshold (default 3). Entries with N ≥ 5 warrant flagging to a human reviewer — independence analysis requires domain judgment.

In [ ]:
SHARED_MIN = 3  # minimum number of distinct sections citing a target

inverted = _parse_refs(DOCUMENT)

token_sections: dict = {}
for compound_tgt, locs in inverted.items():
    for token in _split_citation(compound_tgt):
        for _, section in locs:
            token_sections.setdefault(token, set()).add(section)

candidates = {t: s for t, s in token_sections.items() if len(s) >= SHARED_MIN}

if not candidates:
    print(f"No shared premises found (threshold: {SHARED_MIN}+ distinct sections).")
else:
    sorted_candidates = sorted(candidates.items(), key=lambda x: -len(x[1]))
    print(f"Shared premises: {len(candidates)} target(s) cited from {SHARED_MIN}+ distinct sections")
    print()
    print(f"  {'N':>3}  Target")
    print(f"  {'─'*3}  {'─'*70}")
    for token, sections in sorted_candidates:
        n = len(sections)
        flag = "  *** flag for review" if n >= 5 else ""
        print(f"  {n:>3}  {token}{flag}")
        for sec in sorted(sections):
            print(f"       ↳ {sec}")
        print()

## Load-Bearing Nodes

Which non-heading nodes are most structurally connected? High degree centrality means many other nodes depend on or support this one — changes here have the widest blast radius.

In [ ]:
TOP_N = 10

centrality = nx.degree_centrality(G)
ranked = sorted(
    [(n, s) for n, s in centrality.items() if G.nodes[n].get("type") != "heading"],
    key=lambda x: x[1], reverse=True
)[:TOP_N]

print(f"{'#':>3}  {'Centrality':>10}  {'Line':>4}  {'Node ID':<50}  Topic")
print("-" * 110)
for i, (n, score) in enumerate(ranked, 1):
    attrs = G.nodes[n]
    label = (attrs.get("topic") or attrs.get("lead_sentence", ""))[:40]
    print(f"{i:>3}  {score:>10.4f}  {attrs.get('line',0):>4}  {n:<50}  {label}")

## T2 MECE Scaffolding

Direct children of each heading, grouped for T2 MECE review.  
This cell surfaces the sibling sets — it does **not** evaluate MECE.  
MECE requires a domain-expert human auditor: mutual exclusivity and collective exhaustiveness cannot be determined from structure alone.

Change `MECE_SLUG` to scope to one section, or set to `None` for the full document.

In [ ]:
MECE_SLUG = None  # e.g. "h2:1----" to scope to one section, or None for all

topo = list(nx.topological_sort(G))
headings = [n for n in topo if G.nodes[n].get("type") == "heading"]

if MECE_SLUG:
    headings = [h for h in headings if MECE_SLUG in h
                or any(MECE_SLUG in a for a in nx.ancestors(G, h))]

for h in headings:
    children = [v for v in G.successors(h) if G.nodes[v].get("type") != "heading"]
    if not children:
        continue
    level  = G.nodes[h].get("level", 0)
    label  = G.nodes[h].get("lead_sentence", h)[:70]
    indent = "  " * (level - 1)
    flag   = "N/A (single item)" if len(children) == 1 else "→ T2 MECE review"
    print(f"{indent}{label}")
    print(f"{indent}  Node: {h}  |  Children: {len(children)}  |  {flag}")
    for child in sorted(children, key=lambda x: G.nodes[x].get("line", 0)):
        attrs = G.nodes[child]
        ctype = attrs.get("content_type", "Unknown")
        topic = (attrs.get("topic") or attrs.get("lead_sentence", ""))[:55]
        print(f"{indent}    [{ctype:<10}] line {attrs.get('line',0):>4}  {topic}")
    print()

## Interactive Graph Visualisation (pyvis)

Renders the graph (or a subgraph) as a self-contained HTML file with pan, zoom, and drag.

Requires: `pip install pyvis`  
Change `VIS_SLUG` to scope to a section, or set to `None` to render the full graph.

In [ ]:
VIS_SLUG = "h2:1----"   # set to None for full graph
OUTPUT_HTML = "audit_graph.html"  # saved relative to Jupyter working directory

try:
    from pyvis.network import Network
except ImportError:
    print("pyvis not found — run: pip install pyvis")
    raise

TYPE_COLOURS = {
    "Argument": "#4e79a7",
    "Scope":    "#f28e2b",
    "Claim":    "#e15759",
    "Closure":  "#76b7b2",
    "Unknown":  "#bab0ac",
}

if VIS_SLUG:
    candidates = [n for n in G.nodes() if G.nodes[n].get("type") == "heading" and VIS_SLUG in n]
    if len(candidates) != 1:
        print(f"Expected 1 match for '{VIS_SLUG}', got {len(candidates)} — rendering full graph.")
        render_G = G
    else:
        root = candidates[0]
        scope = {root} | nx.descendants(G, root)
        render_G = G.subgraph(scope)
else:
    render_G = G

# cdn_resources='in_line' embeds all JS inline — avoids Chrome/Safari display
# issues with cdn_resources='local' in notebook environments.
net = Network(height="750px", width="100%", directed=True, notebook=True,
              cdn_resources="in_line")
for n, attrs in render_G.nodes(data=True):
    label = (attrs.get("topic") or attrs.get("lead_sentence", n))[:40]
    colour = TYPE_COLOURS.get(attrs.get("content_type", "Unknown"), "#bab0ac")
    tooltip = f"Line {attrs.get('line','')}\n{attrs.get('text','')[:200]}"
    net.add_node(n, label=label, color=colour, title=tooltip)
for u, v, data in render_G.edges(data=True):
    net.add_edge(u, v, label=data.get("relation", ""))

net.save_graph(OUTPUT_HTML)
print(f"Saved: {Path('.').resolve() / OUTPUT_HTML}")

from IPython.display import IFrame
IFrame(OUTPUT_HTML, width="100%", height=800)